# CredResolve Collections Analysis

The question: is "recovery improved 11% month-on-month" actually true, and where should the
next ₹10 Cr go? This notebook is the actual working-through, not just the final charts —
every claim below is something I checked, not something I assumed.

Run the cells in order — the first one builds everything from the raw CSVs, nothing else
needs to be pre-built.

In [1]:
import duckdb
import pandas as pd
pd.set_option('display.width', 140)

DATA_DIR = "../data"

con = duckdb.connect("collections.duckdb")

raw_tables = ["borrowers","accounts","agents","agent_sessions","campaigns","daily_targeting",
"calls","call_attempts","call_dispositions","whatsapp_events","sms_events","field_visits",
"promises_to_pay","payments","vendor_telephony","complaints","account_status_history"]

for t in raw_tables:
    con.execute(f"DROP TABLE IF EXISTS {t}")
    con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('{DATA_DIR}/{t}.csv')")

# Build the golden layer + metrics layer straight from the SQL repo
con.execute(open("../sql/01_golden_dataset.sql").read())
con.execute(open("../sql/02_metrics.sql").read())

print("Loaded", len(raw_tables), "raw tables and built the golden + metrics layer.")

Loaded 17 raw tables and built the golden + metrics layer.


## 1. What's actually in this data

The "12 months" turned out to be more like 7.5 — Jan 1 to Aug 12, 2026, with August cut off
mid-month. All accounts were opened before this window (2024 through Nov 2025), so no new
loans enter mid-analysis, which at least makes "did the portfolio change" easier to check.

In [2]:
con.execute("SELECT 'calls' t, min(event_at) mn, max(event_at) mx FROM calls "
            "UNION ALL SELECT 'payments', min(event_at), max(event_at) FROM payments "
            "UNION ALL SELECT 'accounts_opened', min(opened_at), max(opened_at) FROM accounts").df()

,t,mn,mx
0,calls,2025-12-29 06:52:37,2026-08-12 15:43:05
1,payments,2026-01-01 00:14:40,2026-08-08 23:50:23
2,accounts_opened,2024-01-01 00:02:27,2025-11-30 23:52:36


## 2. Data forensics

Went through all seven things the brief asks about. Full writeup's in the data quality
report — here are the two that mattered most.

### 2.1 Two "duplicate" problems, and they're not the same problem

500 payment IDs are exact copies — same everything, looks like a retry bug. Safe to drop.

But 3,746 payment reference numbers are reused across completely unrelated payments —
different account, different amount, sometimes 200 days apart. This one's a trap: dedupe on
this field and you delete real payments, which is exactly what I almost did before checking
the actual rows.

In [3]:
# True duplicates (safe to drop)
con.execute('''
    SELECT count(*) n_dupe_payment_ids, sum(c-1) extra_rows
    FROM (SELECT payment_id, count(*) c FROM payments GROUP BY 1 HAVING count(*)>1) d
''').df()

,n_dupe_payment_ids,extra_rows
0,500,500.0


In [4]:
# Reference collision trap — sample proof these are UNRELATED payments sharing a reference
con.execute('''
    SELECT payment_reference, count(*) n_rows, count(DISTINCT account_id) distinct_accounts,
           count(DISTINCT amount) distinct_amounts,
           date_diff('day', min(event_at), max(event_at)) day_span
    FROM payments
    WHERE payment_reference IS NOT NULL
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n_rows DESC LIMIT 5
''').df()

,payment_reference,n_rows,distinct_accounts,distinct_amounts,day_span
0,TXN0000065723,5,4,4,71
1,TXN0000050468,5,3,3,136
2,TXN0000006936,5,4,4,150
3,TXN0000044312,5,5,5,209
4,TXN0000002005,5,4,4,152


### 2.2 Identity fields are scrambled — agents and borrower_id both

The agents table has 30,000 rows for 1,000 agent IDs, and the name/team attached to one agent
ID changes almost every row. Then I checked borrower_id across every event table against the
accounts table (the one place it should be reliable) — 0.0% match rate, all ten tables. So
account_id is the only ID in this whole dataset you can actually trust for joining anything.

In [5]:
for tbl in ["calls","payments","promises_to_pay","field_visits","complaints"]:
    r = con.execute(f'''
        SELECT round(100.0*count(*) FILTER (WHERE t.borrower_id = a.borrower_id)/count(*), 2) AS pct_match
        FROM {tbl} t JOIN accounts a ON t.account_id = a.account_id
    ''').fetchone()[0]
    print(f"{tbl:20s} borrower_id match rate vs accounts master: {r}%")

calls                borrower_id match rate vs accounts master: 0.01%
payments             borrower_id match rate vs accounts master: 0.0%
promises_to_pay      borrower_id match rate vs accounts master: 0.01%
field_visits         borrower_id match rate vs accounts master: 0.01%
complaints           borrower_id match rate vs accounts master: 0.01%


### 2.3 Only 17% of payments trace back to a campaign

Tried joining payments to campaign-targeting records within 30 days. Matched 3,012 of 17,880
successful payments — under 17%. Anything I say below about which channel "works better" is
really only about that 17%, not the whole picture.

In [6]:
con.execute('''
    WITH matched AS (
        SELECT DISTINCT p.payment_id FROM payments p
        JOIN daily_targeting dt ON p.account_id = dt.account_id
        WHERE p.payment_status='SUCCESS'
          AND p.event_at BETWEEN dt.target_date AND dt.target_date + INTERVAL 30 DAY
    )
    SELECT
        (SELECT count(*) FROM payments WHERE payment_status='SUCCESS') AS total_success,
        (SELECT count(*) FROM matched) AS matched_to_campaign,
        round(100.0*(SELECT count(*) FROM matched) /
              (SELECT count(*) FROM payments WHERE payment_status='SUCCESS'), 1) AS pct_attributable
''').df()

,total_success,matched_to_campaign,pct_attributable
0,17880,3012,16.8


## 3. Is the 11% real?

Rebuilt monthly recovery from the cleaned payments (real duplicates gone, reference-collision
trap avoided) and looked at it three ways.

In [7]:
monthly = con.execute('''
    SELECT date_trunc('month', event_at) mo,
           sum(amount) FILTER (WHERE payment_status='SUCCESS') amt_recovered
    FROM golden_payments
    WHERE event_at < '2026-08-01'
    GROUP BY 1 ORDER BY 1
''').df()
monthly['pct_change_mom'] = monthly['amt_recovered'].pct_change() * 100
monthly

,mo,amt_recovered,pct_change_mom
0,2026-01-01,1.872291e+08,NaN
1,2026-02-01,1.701425e+08,-9.126077
2,2026-03-01,1.889124e+08,11.031885
3,2026-04-01,1.751380e+08,-7.291386
4,2026-05-01,1.842503e+08,5.202887
5,2026-06-01,1.755597e+08,-4.716710
6,2026-07-01,1.872423e+08,6.654452


Feb to March alone: +11.11%. That's the number. Jan to March, two months: +1.05%. Jan to July,
the whole window: basically 0%. January and July recovered almost the same amount. The 11%
is real, it's just one data point standing in for a trend that isn't there.

## 4. Why did it happen

Checked every driver the brief lists — portfolio mix, DPD, agent, channel, calling time,
attempt frequency. Short answer: nothing moved. Contact rate, PTP rate, PTP-kept-rate,
segment mix, per-channel conversion — all flat across the seven months, no real trend
anywhere.

In [8]:
con.execute("SELECT mo, contact_rate_pct, ptp_rate_pct, ptp_kept_rate_pct, recovered_gross, "
            "recovery_per_agent_hour FROM monthly_metrics WHERE mo < '2026-08-01'").df()

,mo,contact_rate_pct,ptp_rate_pct,ptp_kept_rate_pct,recovered_gross,recovery_per_agent_hour
0,2025-12-01,0.00,NaN,NaN,NaN,NaN
1,2026-01-01,20.05,50.66,48.52,1.872291e+08,16773.47
2,2026-02-01,19.70,49.80,50.72,1.701425e+08,16116.60
3,2026-03-01,19.99,51.33,49.60,1.889124e+08,16972.22
4,2026-04-01,19.30,52.00,50.08,1.751380e+08,16491.42
5,2026-05-01,20.34,50.34,49.30,1.842503e+08,17008.11
6,2026-06-01,20.40,50.59,49.43,1.755597e+08,16403.03
7,2026-07-01,19.34,53.98,49.12,1.872423e+08,16747.99


### 4.1 Every channel converts about the same, costs are what differ

On the 17% I can actually attribute: conversion is ~8% regardless of channel, and account
difficulty (DPD, NPA%) assigned to each channel is balanced too — so it's not that Field gets
harder cases. WhatsApp's higher total recovery earlier is just volume — more touches, same
conversion rate per touch.

In [9]:
con.execute('''
    WITH touched AS (
        SELECT camp.channel, dt.account_id,
               CASE WHEN p.payment_id IS NOT NULL THEN 1 ELSE 0 END converted
        FROM golden_daily_targeting dt
        JOIN golden_campaigns camp ON dt.campaign_id = camp.campaign_id
        LEFT JOIN golden_payments p ON p.account_id = dt.account_id AND p.payment_status='SUCCESS'
            AND p.event_at BETWEEN dt.target_date AND dt.target_date + INTERVAL 30 DAY
    )
    SELECT channel, count(*) n_touches, round(100.0*avg(converted),2) conversion_rate_pct
    FROM touched GROUP BY 1 ORDER BY 2 DESC
''').df()

,channel,n_touches,conversion_rate_pct
0,WHATSAPP,11450,7.13
1,SMS,10573,7.11
2,MIXED,8769,7.49
3,VOICE,7517,7.57
4,FIELD,6803,7.64


## 5. Counterfactual — what if targeting strategy hadn't changed?

Full methodology (treatment/control groups, assumptions, limitations) is in
`sql/04_counterfactual.sql`. Short version: campaigns don't have one clean before/after
switch date — legacy through v3 strategy all run in parallel — so I used a
difference-in-differences setup, splitting campaigns into an "old" vs "new" strategy
generation and an "early" vs "late" cohort by start date, since that's the best proxy for
before/after available in this data.

In [10]:
con.execute(open("../sql/04_counterfactual.sql").read().split("-- RESULT")[0]).df()

,strategy_gen,cohort,n_touches,conversion_rate_pct
0,NEW,EARLY,10866,7.32
1,NEW,LATE,11173,7.20
2,OLD,EARLY,11750,7.36
3,OLD,LATE,11323,7.50


DiD estimate comes out to -0.26 percentage points — well within the same noise band as
everything else in this dataset. I can't detect a real effect from the targeting-strategy
change with this design. If they hadn't changed strategy, recovery would plausibly look
about the same as what actually happened.

## 6. Investment recommendation

Full reasoning, cost assumptions, and the downside case are in the memo. Short version:
WhatsApp/digital, moderate confidence — every channel converts at about the same rate, so the
one that's cheapest per attempt wins on expected total recovery.

## 7. Everything, sorted by how sure I actually am

| Finding | How sure |
|---|---|
| Jan ≈ Jul recovery (flat trend) | Checked 3 ways, solid |
| 11% = Feb→Mar cherry-pick | Solid |
| ~500 duplicate payment_id rows inflate recovery slightly | Solid |
| payment_reference isn't a safe way to dedupe | Solid |
| borrower_id unusable outside accounts/borrowers (0% match) | Solid |
| agents.csv identity fields unusable | Solid |
| Only 17% of payments are channel-attributable | Solid |
| Portfolio mix / contact rate / PTP rate flat over time | Fairly solid |
| Channel per-touch conversion roughly equal | Correlation, small/biased subsample |
| Targeting-strategy change had no detectable effect | Fairly solid null result |
| WhatsApp is the best ₹10 Cr bet | Best guess, needs the pilot to confirm |

## 8. Second pass — things I caught on a recheck

Went back through and found a few gaps: telephony vendor mapping (one of the seven forensic
checks I'd been light on), geography and calling-attempt frequency as drivers, and an actual
explicit Simpson's-paradox test instead of just implying one.

### 8.1 Telephony vendor mapping — missed this the first time around

In [11]:
con.execute('''
    SELECT v.status AS vendor_status, count(*) n_calls
    FROM golden_calls c JOIN vendor_telephony v ON c.vendor_id = v.vendor_id
    GROUP BY 1
''').df()

,vendor_status,n_calls
0,ACTIVE,35911
1,INACTIVE,54089


60% of all calls (54,089 of 90,000) go through vendors currently marked "inactive" — and not
as some old tail end, this runs right through to the last days of data (Aug 8-12). The status
field in the vendor table just doesn't reflect what's actually happening operationally.
Didn't use it for anything after finding this.

### 8.2 Geography and how many times an account gets called

In [12]:
con.execute('''
    SELECT attempt_no, count(*) n,
           round(100.0*avg(CASE WHEN EXISTS(
               SELECT 1 FROM golden_payments p WHERE p.account_id=ca.account_id
               AND p.payment_status='SUCCESS'
               AND p.event_at BETWEEN ca.event_at AND ca.event_at + INTERVAL 30 DAY
           ) THEN 1 ELSE 0 END), 2) AS pct_later_paid
    FROM golden_call_attempts ca WHERE attempt_no <= 7 GROUP BY 1 ORDER BY 1
''').df()

,attempt_no,n,pct_later_paid
0,1,17089,6.74
1,2,17100,7.20
2,3,17201,7.41
3,4,17252,6.94
4,5,16952,7.03
5,6,17207,7.17
6,7,17199,7.09


Conversion odds stay flat (6.7-7.4%) no matter which attempt number reached the account —
calling more times isn't doing much here. Geography tracks account volume, nothing more
interesting than that. `client` and `language`, which the brief asks about, don't exist
anywhere in the 17 source tables — nothing to check, they're just not present.

### 8.3 The Simpson's paradox check

In [13]:
con.execute('''
    WITH t AS (
        SELECT date_trunc('month', dt.target_date) mo, a.risk_segment,
               CASE WHEN p.payment_id IS NOT NULL THEN 1 ELSE 0 END converted
        FROM golden_daily_targeting dt
        JOIN golden_accounts a ON dt.account_id = a.account_id
        LEFT JOIN golden_payments p ON p.account_id=dt.account_id AND p.payment_status='SUCCESS'
            AND p.event_at BETWEEN dt.target_date AND dt.target_date + INTERVAL 30 DAY
        WHERE date_trunc('month',dt.target_date) IN ('2026-01-01','2026-07-01')
    )
    SELECT risk_segment, mo, round(100.0*avg(converted),2) conv_pct, count(*) n
    FROM t GROUP BY 1,2 ORDER BY 1,2
''').df()

,risk_segment,mo,conv_pct,n
0,HIGH,2026-01-01,7.23,1604
1,HIGH,2026-07-01,5.26,1579
2,LOW,2026-01-01,7.83,1623
3,LOW,2026-07-01,5.37,1564
4,MEDIUM,2026-01-01,7.15,1581
5,MEDIUM,2026-07-01,6.39,1564
6,NPA,2026-01-01,8.32,1575
7,NPA,2026-07-01,6.96,1537


Every single risk segment shows a real decline from Jan to Jul — NPA 8.32% down to 6.96%, LOW
7.83% down to 5.37%, and so on. That's not actually a contradiction of the flat-recovery
finding above — this is measured on the 17%-attributable subset, the other one uses all
payments — but it's the clearest evidence that the attribution gap is hiding something real.
If I had more time, fixing that would be priority one before the next budget cycle.